# Labor 3: Informationstheorie im Notebook

**Name:**  
**Datum:**  

Bearbeiten Sie die TODO-Stellen. Führen Sie am Ende alle Zellen von oben nach unten aus und speichern Sie das Notebook für Moodle.

Ziel des Labors: Sie vergleichen Textdaten, Zufallsdaten, Entropie und Codewortlängen so, dass die Grenze verlustfreier Kompression praktisch sichtbar wird.


## 1. Setup

Die erste Zelle lädt nur Bibliotheken und Hilfsfunktionen. Hier müssen Sie noch nichts ändern.


In [ ]:
import gzip
import math
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

def gzip_bytes(data: bytes) -> bytes:
    # Komprimiert die Daten mit gzip und gibt die komprimierten Bytes zurück
    return gzip.compress(data)

def byte_counts(data: bytes):
    # Zählt die Häufigkeit jedes Bytewerts (0-255) in den Daten und gibt ein Array mit den Zählungen zurück
    values = np.frombuffer(data, dtype=np.uint8)
    return np.bincount(values, minlength=256)

def byte_probabilities(data: bytes):
    # Berechnet die relativen Häufigkeiten der Bytewerte in den Daten und gibt ein Array mit den Wahrscheinlichkeiten zurück
    counts = byte_counts(data)
    return counts / counts.sum()

def show_size_report(label, original, compressed):
    # Zeigt einen Bericht über die Größe der Original- und Komprimierten Daten sowie das Verhältnis an
    ratio = len(compressed) / len(original) * 100
    print(f"{label}")
    print(f"  Original:    {len(original):>8} Byte")
    print(f"  Komprimiert: {len(compressed):>8} Byte")
    print(f"  Verhältnis:  {ratio:>8.2f} %")

def plot_byte_histogram(data: bytes, title: str, ax=None):
    # Plottet ein Histogramm der Bytewerte in den Daten. Wenn kein Achsenobjekt übergeben wird, wird ein neues erstellt.
    if ax is None:
        _, ax = plt.subplots(figsize=(9, 3))
    probs = byte_probabilities(data)
    ax.bar(np.arange(256), probs, width=1.0)
    ax.set_title(title)
    ax.set_xlabel("Bytewert")
    ax.set_ylabel("relative Häufigkeit")
    ax.set_xlim(0, 255)
    return ax


## 2. Textdaten komprimieren

Der Beispieltext ist absichtlich regelmäßig. Ergänzen Sie die Berechnung des Kompressionsverhältnisses.


In [ ]:
# load text data from /data/big.txt into a string variable
with open("./data/big.txt", "r", encoding="utf-8") as f:
    basis_text = f.read()

text_data = (basis_text).encode("utf-8")
text_gzip = gzip_bytes(text_data)

# TODO: Ergänzen Sie die Formel für das Verhältnis in Prozent. Anzahl der Bytes in einer Variablen erhalten Sie mit len(variable)
kompressionsverhaeltnis_text = ...

show_size_report("Textdaten", text_data, text_gzip)
print(f"Selbst berechnetes Verhältnis: {kompressionsverhaeltnis_text:.2f} %")


**Kurze Deutung:**  
Warum schrumpft der Text? Ergänzen Sie 2 bis 3 Stichpunkte.

- ...
- ...


## 3. Zufallsdaten vergleichen

Nun werden Zufallsdaten mit derselben Länge erzeugt. Ergänzen Sie wieder das Kompressionsverhältnis.


In [ ]:
random_data = rng.integers(0, 256, size=len(text_data), dtype=np.uint8).tobytes()
random_gzip = gzip_bytes(random_data)

# TODO: Ergänzen Sie die Formel für das Verhältnis in Prozent.
kompressionsverhaeltnis_zufall = ...

show_size_report("Zufallsdaten", random_data, random_gzip)
print(f"Selbst berechnetes Verhältnis: {kompressionsverhaeltnis_zufall:.2f} %")


**Kurze Deutung:**  
Vergleichen Sie Text und Zufallsdaten. Verwenden Sie die Begriffe `Struktur`, `Vorhersagbarkeit` und `Redundanz`.

> ...


## 4. Verteilungen sichtbar machen

Histogramme zeigen, ob bestimmte Bytewerte deutlich häufiger auftreten als andere.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3), sharey=True)
plot_byte_histogram(text_data, "Byte-Häufigkeiten im Text", axes[0])
plot_byte_histogram(random_data, "Byte-Häufigkeiten in Zufallsdaten", axes[1])
plt.tight_layout()
plt.show()


In [ ]:
# TODO: Geben Sie die 10 häufigsten Bytewerte im Text aus.
# Hinweis: Counter(text_data).most_common(10)
haeufigste_textbytes = ...
haeufigste_textbytes

# Was sind das für Zeichen als UTF-8-codierte Bytewerte? Sie können die Bytewerte mit bytes([byte_value]).decode("utf-8") in Zeichen umwandeln.
haeufigste_textzeichen = ...
haeufigste_textzeichen

**Kurze Deutung:**  
Welche Verteilung wirkt stärker strukturiert? Welche Quelle erwarten Sie mit höherer Entropie?

> ...


## 5. Informationsgehalt

Der Informationsgehalt eines Ereignisses lautet

$$
I(x_i)=-\log_2(P(x_i)).
$$

Ergänzen Sie die Funktion.


In [ ]:
def informationsgehalt(p):
    if p <= 0:
        return math.inf
    I = -math.log2(p)
    return I

wahrscheinlichkeiten = [1.0, 0.5, 0.25, 0.1]

# TODO: Ergänzen Sie eine eigene Wahrscheinlichkeit zwischen 0 und 1.
wahrscheinlichkeiten.append(...)

for p in wahrscheinlichkeiten:
    print(f"p={p:>5}: I={informationsgehalt(p):.3f} bit")


**Kurze Deutung:**  
Was passiert mit dem Informationsgehalt, wenn die Wahrscheinlichkeit von `0.5` auf `0.25` sinkt?

> ...


## 6. Entropie von Quellen

Entropie ist der mittlere Informationsgehalt einer Quelle:

$$
H(X)=-\sum_i p_i\log_2(p_i).
$$

Ergänzen Sie die Funktion und eine eigene Quelle.


In [ ]:
def entropie(probs):
    probs = np.array(probs, dtype=float)
    probs = probs[probs > 0]
    entropie = -np.sum(probs * np.log2(probs)) 
    return entropie

quellen = {
    "faire Münze": [0.5, 0.5],
    "gezinkte Münze": [0.9, 0.1],
    "vier gleiche Zeichen": [0.25, 0.25, 0.25, 0.25],
    # TODO: Ersetzen Sie die Werte durch eine eigene gültige Verteilung.
    "eigene Quelle": ...
}

for name, probs in quellen.items():
    print(f"{name:>22}: H={entropie(probs):.3f} bit/Zeichen")


In [ ]:
# Wenn die Entropiefunktion funktioniert, können auch die Byte-Verteilungen verglichen werden.
H_text_bytes = entropie(byte_probabilities(text_data))
H_random_bytes = entropie(byte_probabilities(random_data))

print(f"Textdaten:    {H_text_bytes:.3f} bit/Byte")
print(f"Zufallsdaten: {H_random_bytes:.3f} bit/Byte")


**Kurze Deutung:**  
Welche Quelle hat die höchste Entropie? Welche Quelle sollte am besten komprimierbar sein?

> ...


## 7. Huffman-Code auswerten

Für die Quelle aus der Vorlesung wird eine mögliche Huffman-Codetabelle vorgegeben. Prüfen Sie Präfixfreiheit, mittlere Codewortlänge und Abstand zur Entropie.


In [ ]:
symbole = ["R", "D", "N", "Z", "L"]
wahrscheinlichkeiten_huffman = np.array([0.38, 0.23, 0.10, 0.09, 0.20])
codes = {
    "R": "10",
    "D": "11",
    "L": "01",
    "N": "001",
    "Z": "000",
}

def ist_praefixfrei(code_dict):
    codewoerter = list(code_dict.values())
    for a in codewoerter:
        for b in codewoerter:
            if a != b and b.startswith(a):
                return False
    return True

def mittlere_codewortlaenge(symbole, probs, code_dict):
    # TODO: Berechnen Sie L=sum_i p_i*l_i.
    L = ...
    return L

H = entropie(wahrscheinlichkeiten_huffman)
L = mittlere_codewortlaenge(symbole, wahrscheinlichkeiten_huffman, codes)

print("präfixfrei:", ist_praefixfrei(codes))
print(f"H(X) = {H:.3f} bit/Zeichen")
print(f"L    = {L:.3f} bit/Zeichen")
print(f"Abstand L-H = {L-H:.3f} bit/Zeichen")


**Kurze Deutung:**  
Warum liegt `L` nahe bei `H(X)`, aber nicht darunter?

> ...


## 8. Ergebnisreflexion für Moodle

Fassen Sie die wichtigsten Ergebnisse knapp zusammen.

1. Textdaten schrumpfen stärker als Zufallsdaten, weil ...
2. Entropie beschreibt ...
3. Die höchste Entropie im Notebook hatte ...
4. Die niedrigste Entropie im Notebook hatte ...
5. Der Huffman-Code zeigt, dass ...
6. Eine mögliche Klausurfrage zu diesem Labor wäre ...


## 9. Abgabe

Speichern Sie dieses Notebook nach der Bearbeitung. Geben Sie in Moodle ab:

- die ausgefüllte `.ipynb`-Datei
- optional zusätzlich einen HTML- oder PDF-Export

Vor der Abgabe: `Kernel` neu starten und alle Zellen von oben nach unten ausführen.
